In [ ]:
!pip install timm faiss-cpu scikit-learn opencv-python huggingface_hub --quiet

In [ ]:
import os
import random
import time
import numpy as np
import cv2
import json
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import timm
import faiss
from sklearn.metrics import roc_auc_score, precision_recall_curve
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

# Đăng nhập Hugging Face an toàn qua Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Đăng nhập Hugging Face thành công!")
except Exception as e:
    print("⚠️ Chưa tìm thấy HF_TOKEN trong Kaggle Secrets. Quá trình lưu mô hình lên HF có thể bị bỏ qua.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Đang sử dụng thiết bị: {device}")

CONFIG = {
    'data_path': '/kaggle/input/datasets/ipythonx/mvtec-ad', 
    'image_size': 224,
    'batch_size': 64,          # BÀI BÁO: "the batch size was set to 128"
    'coreset_ratio': 0.1,    
    'fine_tune_epochs': 100,    # BÀI BÁO: "the maximum epoch was set as 100"
    'learning_rate': 1e-4,
    'hf_repo_id': "calmm-m/block1" 
}

try:
    api = HfApi()
    api.create_repo(repo_id=CONFIG['hf_repo_id'], repo_type="model", exist_ok=True)
except:
    pass

MVTEC_CATEGORIES = [
    'carpet', 'grid', 'leather', 'tile', 'wood',
    'bottle', 'cable', 'capsule', 'hazelnut', 'metal_nut', 
    'pill', 'screw', 'toothbrush', 'transistor', 'zipper'
]

In [ ]:
class CutPasteAugmentation:
    def __init__(self, patch_size_ratio=(0.05, 0.2)):
        self.patch_size_ratio = patch_size_ratio

    def __call__(self, image):
        img_np = np.array(image)
        h, w, c = img_np.shape
        
        patch_area = h * w * random.uniform(self.patch_size_ratio[0], self.patch_size_ratio[1])
        patch_aspect = random.uniform(0.5, 2.0)
        patch_h = min(int(np.sqrt(patch_area * patch_aspect)), h-1)
        patch_w = min(int(np.sqrt(patch_area / patch_aspect)), w-1)

        crop_y = random.randint(0, h - patch_h)
        crop_x = random.randint(0, w - patch_w)
        patch = img_np[crop_y:crop_y+patch_h, crop_x:crop_x+patch_w, :].copy()

        angle = random.uniform(-45, 45)
        M = cv2.getRotationMatrix2D((patch_w/2, patch_h/2), angle, 1)
        patch_rotated = cv2.warpAffine(patch, M, (patch_w, patch_h))

        paste_y = random.randint(0, h - patch_h)
        paste_x = random.randint(0, w - patch_w)
        
        aug_img = img_np.copy()
        aug_img[paste_y:paste_y+patch_h, paste_x:paste_x+patch_w, :] = patch_rotated
        return Image.fromarray(aug_img)

class MVTecDataset(Dataset):
    def __init__(self, root_dir, category, is_train=True, use_cutpaste=False):
        self.image_paths = []
        self.labels = []
        self.mask_paths = []
        self.use_cutpaste = use_cutpaste
        self.cutpaste_aug = CutPasteAugmentation()
        
        self.transform = T.Compose([
            T.Resize((256, 256)),          
            T.CenterCrop((224, 224)),      
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.mask_transform = T.Compose([
            T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST),
            T.CenterCrop((224, 224)),
            T.ToTensor()
        ])

        if is_train:
            img_dir = os.path.join(root_dir, category, 'train', 'good')
            for img_name in os.listdir(img_dir):
                if img_name.endswith(('.png', '.jpg')):
                    self.image_paths.append(os.path.join(img_dir, img_name))
                    self.labels.append(0) 
                    self.mask_paths.append(None) # Ảnh bình thường không có mask
        else:
            test_dir = os.path.join(root_dir, category, 'test')
            for defect_type in os.listdir(test_dir):
                defect_dir = os.path.join(test_dir, defect_type)
                if not os.path.isdir(defect_dir): continue
                label = 0 if defect_type == 'good' else 1
                for img_name in os.listdir(defect_dir):
                    if img_name.endswith(('.png', '.jpg')):
                        self.image_paths.append(os.path.join(defect_dir, img_name))
                        self.labels.append(label)
                        # THÊM MỚI: Nạp đường dẫn Mask cho tập Test
                        if label == 1:
                            mask_name = img_name.rsplit('.', 1)[0] + '_mask.png'
                            mask_path = os.path.join(root_dir, category, 'ground_truth', defect_type, mask_name)
                            self.mask_paths.append(mask_path)
                        else:
                            self.mask_paths.append(None)

    def __len__(self):
        return len(self.image_paths) * (2 if self.use_cutpaste else 1)

    def __getitem__(self, idx):
        img_idx = idx // 2 if self.use_cutpaste else idx
        
        if self.use_cutpaste:
            is_anomaly = (idx % 2 == 1)
        else:
            is_anomaly = self.labels[img_idx]

        image = Image.open(self.image_paths[img_idx]).convert('RGB')
        
        if self.use_cutpaste and is_anomaly:
            image = self.cutpaste_aug(image)
            label = 1
        elif self.use_cutpaste:
            label = 0
        else:
            label = is_anomaly
            
        # THÊM MỚI: Xử lý Mask
        if self.mask_paths[img_idx] is not None and not self.use_cutpaste:
            mask = Image.open(self.mask_paths[img_idx]).convert('L')
            mask = self.mask_transform(mask)
        else:
            mask = torch.zeros((1, 224, 224)) # Ảnh bình thường hoặc CutPaste tạo mask trống
            
        return self.transform(image), label, mask

In [ ]:
class ViTCoreExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # Trọng số pre-trained ImageNet
        self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=2)
        
        # ❌ ĐÃ XÓA: self.avg_pool = nn.AvgPool2d(...)
        self._register_hooks()

    def _register_hooks(self):
        def hook_fn(module, input, output):
            # Gắn trực tiếp kết quả vào module hiện tại đang chạy trên GPU đó
            module.extracted_feature = output
            
        # 🟢 CHỈ SỬ DỤNG 1 HOOK: Chọn Block 3 của Stage 3 (hoặc bạn có thể đổi thành blocks[2] tùy ý)
        self.backbone.layers[2].blocks[0].register_forward_hook(hook_fn)

    def forward_features(self, x):
        # Chạy ảnh qua mạng
        _ = self.backbone.forward_features(x)
        
        # Lấy đặc trưng trực tiếp từ Block duy nhất đã đăng ký hook
        feat = self.backbone.layers[2].blocks[0].extracted_feature
        
        # Swin trả về [B, H, W, C], ta đổi trục về chuẩn PyTorch [B, C, H, W]
        block_feat = feat.permute(0, 3, 1, 2)
        
        # 🟢 TRẢ VỀ TRỰC TIẾP (Không cần Pooling, không cần torch.cat)
        return block_feat
    
    def forward_classifier(self, x):
        return self.backbone(x)

In [ ]:
def k_center_greedy(features, ratio):
    num_samples = int(features.shape[0] * ratio)
    if num_samples == 0: return features
    coreset_idx = [np.random.randint(0, features.shape[0])]
    min_distances = np.linalg.norm(features - features[coreset_idx[0]], axis=1)
    for _ in range(1, num_samples):
        farthest_idx = np.argmax(min_distances)
        coreset_idx.append(farthest_idx)
        new_distances = np.linalg.norm(features - features[farthest_idx], axis=1)
        min_distances = np.minimum(min_distances, new_distances)
    return features[coreset_idx]

def calculate_anomaly_scores(test_features, memory_bank_index, k=9):
    B, C, H, W_dim = test_features.shape # Dùng W_dim để không trùng với biến trọng số W
    
    # 1. Đập dẹp ảnh test thành các mảnh
    test_patches = test_features.view(B, C, H * W_dim).permute(0, 2, 1).reshape(-1, C).cpu().numpy()
    
    # 2. Tìm K láng giềng gần nhất trong Memory Bank
    # Kết quả 'distances' sẽ là Tensor 2 chiều có dạng [B * 196, K] (ví dụ: [196, 9])
    distances, _ = memory_bank_index.search(test_patches, k)
    
    # FAISS trả về khoảng cách L2 bình phương, ta lấy căn bậc 2 để ra khoảng cách Euclid thực
    distances = np.sqrt(distances)
    distances_tensor = torch.tensor(distances)
    
    # 3. Áp dụng Softmax trên trục K-láng giềng (dim=1) 
    # Tính toán sự phân bố của 9 láng giềng cho TỪNG MẢNH ẢNH.
    softmax_weights = F.softmax(distances_tensor, dim=1)
    
    # "Score" gốc là khoảng cách đến láng giềng sát nhất (cột 0) [cite: 374]
    base_scores = distances_tensor[:, 0]
    
    # 4. ÁP DỤNG CÔNG THỨC 2: W = 1 - Softmax(D) 
    # Lấy trọng số Softmax của láng giềng sát nhất
    W = 1.0 - softmax_weights[:, 0]
    
    # 5. ÁP DỤNG CÔNG THỨC 3: Anomaly Score = Score * W 
    anomaly_scores_flat = base_scores * W
    
    # Khôi phục lại không gian [B, 196] để chia cho từng bức ảnh trong Batch
    anomaly_scores = anomaly_scores_flat.view(B, H * W_dim)
    
    # Giá trị lớn nhất của 196 mảnh là điểm của cả bức ảnh [cite: 375]
    image_scores = anomaly_scores.max(dim=1)[0].numpy()
    
    # Phục hồi dạng lưới 14x14 cho bản đồ nhiệt [cite: 380]
    patch_scores = anomaly_scores.view(B, H, W_dim).numpy() 
    
    return image_scores, patch_scores

def fine_tune_cutpaste(model, train_loader, epochs):
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    best_loss = float('inf')
    patience = 2
    patience_counter = 0
    
    for epoch in range(epochs):
        total_loss = 0
        for images, labels, _ in train_loader: # THÊM MỚI: Unpack 3 biến do có thêm mask
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model.forward_classifier(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        print(f"      Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0 
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"      -> 🛑 Early Stopping tại epoch {epoch+1}!")
                break
    return model

def build_memory_bank(model, dataloader):
    model.eval()
    memory_bank = []
    with torch.no_grad():
        for images, _, _ in dataloader: # THÊM MỚI: Unpack 3 biến
            features = model.forward_features(images.to(device))
            B, C, H, W = features.shape
            patches = features.view(B, C, H*W).permute(0, 2, 1).reshape(-1, C).cpu().numpy()
            memory_bank.append(patches)
            
    memory_bank = np.concatenate(memory_bank, axis=0)
    coreset = k_center_greedy(memory_bank, CONFIG['coreset_ratio'])
    
    index = faiss.IndexFlatL2(coreset.shape[1])
    index.add(coreset)
    
    return index, coreset

In [ ]:
def find_best_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx]

In [ ]:
import os
import time
import torch
import torch.nn as nn
import json
import faiss
import cv2
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.ndimage import gaussian_filter
from torch.utils.data import DataLoader
import torch.optim as optim

class ViTMultiGPUWrapper(nn.Module):
    """
    Lớp bọc thông minh giúp module nn.DataParallel có thể hiểu và chia đều 
    dữ liệu ra nhiều GPU cho cả 2 hàm chức năng của ViTCoreExtractor.
    """
    def __init__(self, core_model, mode='features'):
        super().__init__()
        self.core_model = core_model
        self.mode = mode

    def forward(self, x):
        if self.mode == 'classifier':
            return self.core_model.forward_classifier(x)
        elif self.mode == 'features':
            return self.core_model.forward_features(x)
        return x

# =====================================================================
# 2. CÁC HÀM XỬ LÝ SONG SONG (MULTI-GPU & MULTI-CPU)
# =====================================================================
def fine_tune_cutpaste_parallel(model, train_loader, epochs):
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    # Kích hoạt Multi-GPU nếu phát hiện có trên 1 card đồ họa
    if torch.cuda.device_count() > 1:
        parallel_model = nn.DataParallel(ViTMultiGPUWrapper(model, mode='classifier')).to(device) 
    else:
        parallel_model = model
    
    best_loss = float('inf')
    patience = 2
    patience_counter = 0
    
    for epoch in range(epochs):
        total_loss = 0
        for images, labels, _ in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # Tính toán phân tán trên các GPU
            if isinstance(parallel_model, nn.DataParallel):
                outputs = parallel_model(images)
            else:
                outputs = parallel_model.forward_classifier(images)
                
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        print(f"      Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")
        
        # BÀI BÁO: Chỉ cần loss có giảm (dù siêu nhỏ) là reset bộ đếm
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0 
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"      -> 🛑 Early Stopping: Loss không giảm trong {patience} epochs, dừng tại epoch {epoch+1}!")
                break
                
    return model

def build_memory_bank_parallel(model, dataloader):
    model.eval()
    memory_bank = []
    
    # Kích hoạt Multi-GPU cho bộ trích xuất đặc trưng
    if torch.cuda.device_count() > 1:
        parallel_model = nn.DataParallel(ViTMultiGPUWrapper(model, mode='features')).to(device)
    else:
        parallel_model = model
        
    with torch.no_grad():
        for images, _, _ in dataloader:
            if isinstance(parallel_model, nn.DataParallel):
                features = parallel_model(images)
            else:
                features = parallel_model.forward_features(images.to(device))
                
            B, C, H, W = features.shape
            patches = features.view(B, C, H*W).permute(0, 2, 1).reshape(-1, C).cpu().numpy()
            memory_bank.append(patches)
            
    memory_bank = np.concatenate(memory_bank, axis=0)
    coreset = k_center_greedy(memory_bank, CONFIG['coreset_ratio'])
    
    index = faiss.IndexFlatL2(coreset.shape[1])
    index.add(coreset)
    return index, coreset

# =====================================================================
# 3. VÒNG LẶP CHÍNH (MAIN LOOP) CHUẨN THỰC NGHIỆM
# =====================================================================
evaluation_results = {}
num_cpu_cores = os.cpu_count() # Đếm số luồng CPU hiện có của hệ thống

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(device)

for category in MVTEC_CATEGORIES:
    print(f"\n" + "="*85)
    print(f"🚀 XỬ LÝ DANH MỤC: {category.upper()} | CPU Workers: {num_cpu_cores} | GPUs: {torch.cuda.device_count()}")
    print("="*85)
    
    cutpaste_dataset = MVTecDataset(CONFIG['data_path'], category, is_train=True, use_cutpaste=True)
    normal_dataset = MVTecDataset(CONFIG['data_path'], category, is_train=True, use_cutpaste=False)
    test_dataset = MVTecDataset(CONFIG['data_path'], category, is_train=False, use_cutpaste=False)
    
    # Kích hoạt num_workers và pin_memory cho Data Pipeline
    cutpaste_loader = DataLoader(
        cutpaste_dataset, batch_size=CONFIG['batch_size'], shuffle=True, 
        num_workers=num_cpu_cores, pin_memory=True
    )
    normal_loader = DataLoader(
        normal_dataset, batch_size=CONFIG['batch_size'], shuffle=False, 
        num_workers=num_cpu_cores, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, 
        num_workers=num_cpu_cores, pin_memory=True
    )
    
    model = ViTCoreExtractor().to(device)
    
    print(f"[{category}] 1. Đang Fine-tune Cut-Paste ({CONFIG['fine_tune_epochs']} epochs)...")
    model = fine_tune_cutpaste_parallel(model, cutpaste_loader, CONFIG['fine_tune_epochs'])
    
    print(f"[{category}] 2. Xây dựng Memory Bank...")
    faiss_index, coreset_numpy = build_memory_bank_parallel(model, normal_loader)
    
    print(f"[{category}] 3. Suy luận trên tập Test...")
    all_img_scores, all_img_labels = [], []
    all_pixel_scores, all_pixel_labels = [], [] 
    
    start_time = time.time()
    model.eval()
    
    if torch.cuda.device_count() > 1:
        parallel_model = nn.DataParallel(ViTMultiGPUWrapper(model, mode='features')).to(device)
    else:
        parallel_model = model
        
    with torch.no_grad():
        for images, labels, masks in test_loader:
            if isinstance(parallel_model, nn.DataParallel):
                features = parallel_model(images)
            else:
                features = parallel_model.forward_features(images.to(device))
            
            img_scores, patch_scores = calculate_anomaly_scores(features, faiss_index)
            
            all_img_scores.extend(img_scores)
            all_img_labels.extend(labels.numpy())
            
            # Tính toán phân giải Pixel cho từng ảnh trong batch
            for i in range(len(patch_scores)):
                score_map = cv2.resize(patch_scores[i], (224, 224), interpolation=cv2.INTER_LINEAR)
                score_map = gaussian_filter(score_map, sigma=4)
                
                all_pixel_scores.extend(score_map.flatten())
                all_pixel_labels.extend(masks[i].numpy().flatten())
            
    fps = len(test_dataset) / (time.time() - start_time)
    
    # Bóc tách 2 chỉ số AUROC riêng biệt
    image_auroc = roc_auc_score(all_img_labels, all_img_scores)
    pixel_auroc = roc_auc_score(all_pixel_labels, all_pixel_scores) 
    
    print(f"   -> Image AUROC (Phân loại): {image_auroc:.4f}")
    print(f"   -> Pixel AUROC (Khoanh vùng): {pixel_auroc:.4f}")
    print(f"   -> Tốc độ suy luận: {fps:.2f} FPS")
    
    try:
        model_name = f"vit_core_swin_{category}.pth"
        index_name = f"memory_bank_{category}.index"
        metrics_name = f"metrics_{category}.json" # THÊM MỚI: Tên file JSON lưu kết quả
        
        # 1. LƯU LOCAL
        torch.save(model.state_dict(), model_name)
        faiss.write_index(faiss_index, index_name)
        
        # THÊM MỚI: Đóng gói kết quả đánh giá thành file JSON
        cat_metrics = {
            'Image_AUROC': float(image_auroc),
            'Pixel_AUROC': float(pixel_auroc),
            'FPS': float(fps)
        }
        with open(metrics_name, 'w') as f:
            json.dump(cat_metrics, f, indent=4)
        
        # Tính toán dung lượng
        model_size_mb = os.path.getsize(model_name) / (1024 * 1024)
        index_size_mb = os.path.getsize(index_name) / (1024 * 1024)
        peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024) if torch.cuda.is_available() else 0.0

        # 2. PUSH LÊN HUGGING FACE
        print(f"   -> Đang tải toàn bộ dữ liệu lên Hugging Face Repo: {CONFIG['hf_repo_id']}...")
        
        if 'api' in globals():
            # Đẩy Model
            api.upload_file(
                path_or_fileobj=model_name,
                path_in_repo=f"{category}/{model_name}", 
                repo_id=CONFIG['hf_repo_id'], # ĐÃ SỬA: Lấy từ CONFIG
                repo_type="model"
            )
            # Đẩy FAISS Index
            api.upload_file(
                path_or_fileobj=index_name,
                path_in_repo=f"{category}/{index_name}",
                repo_id=CONFIG['hf_repo_id'], # ĐÃ SỬA: Lấy từ CONFIG
                repo_type="model"
            )
            # THÊM MỚI: Đẩy file JSON Metrics
            api.upload_file(
                path_or_fileobj=metrics_name,
                path_in_repo=f"{category}/{metrics_name}",
                repo_id=CONFIG['hf_repo_id'], # ĐÃ SỬA: Lấy từ CONFIG
                repo_type="model"
            )
            print("   -> Upload thành công 3 files (Model, Index, Metrics)! 🎉")
        else:
            print("   -> ⚠️ Bỏ qua upload vì không kết nối được Hugging Face API.")

    except Exception as e:
        print(f"   -> ❌ Quá trình lưu/upload gặp lỗi: {e}")
        model_size_mb, index_size_mb, peak_vram_mb = 0, 0, 0

print("\n" + "*"*105)
print(f"{'BẢNG TỔNG HỢP HIỆU SUẤT VÀ ĐỘ NẶNG TRÊN MVTEC AD (CẤU HÌNH PARALLEL)':^105}")
print("*"*105)
print(f"{'Danh mục':<15} | {'Img AUROC':<10} | {'Pix AUROC':<10} | {'Tốc độ':<10} | {'Mạng nơ-ron':<12} | {'Bộ nhớ FAISS':<12} | {'Max VRAM'}")
print("-" * 105)

avg_img_auroc, avg_pix_auroc, avg_fps, avg_model, avg_index, avg_vram = 0, 0, 0, 0, 0, 0
for cat, metrics in evaluation_results.items():
    print(f"{cat:<15} | {metrics['Img_AUROC']:.4f}     | {metrics['Pix_AUROC']:.4f}     | {metrics['FPS']:>5.1f} FPS  | {metrics['Model_MB']:>7.2f} MB  | {metrics['Index_MB']:>8.2f} MB  | {metrics['VRAM_MB']:>7.2f} MB")
    avg_img_auroc += metrics['Img_AUROC']
    avg_pix_auroc += metrics['Pix_AUROC']
    avg_fps += metrics['FPS']
    avg_model += metrics['Model_MB']
    avg_index += metrics['Index_MB']
    avg_vram = max(avg_vram, metrics['VRAM_MB']) 

n_cats = len(evaluation_results)
print("-" * 105)
print(f"{'TRUNG BÌNH':<15} | {avg_img_auroc/n_cats:.4f}     | {avg_pix_auroc/n_cats:.4f}     | {avg_fps/n_cats:>5.1f} FPS  | {avg_model/n_cats:>7.2f} MB  | {avg_index/n_cats:>8.2f} MB  | {avg_vram:>7.2f} MB")
print("*"*105)